In [0]:
df_customers = spark.table("retailer.silver.customers")

dim_customers = df_customers.select(
    "customer_key",
    "name",
    "gender",
    "age",
    "city",
    "state",
    "country",
    "continent"
).dropDuplicates(["customer_key"])

dim_customers.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("retailer.gold.dim_customers")

In [0]:
df_products = spark.table("retailer.silver.products")

dim_products = df_products.select(
    "product_key",
    "product_name",
    "brand",
    "category",
    "subcategory",
    "color"
).dropDuplicates(["product_key"])

dim_products.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("retailer.gold.dim_products")

In [0]:
df_stores = spark.table("retailer.silver.stores")

dim_stores = df_stores.select(
    "store_key",
    "country",
    "state",
    "square_meters"
).dropDuplicates(["store_key"])

dim_stores.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("retailer.gold.dim_stores")

In [0]:
from pyspark.sql.functions import *

df_sales = spark.table("retailer.silver.sales")

dim_date = df_sales.select(
    col("order_date").alias("date")
).distinct()

dim_date = dim_date.withColumn("year", year("date")) \
    .withColumn("month", month("date")) \
    .withColumn("month_name", date_format("date", "MMMM")) \
    .withColumn("quarter", quarter("date"))

dim_date.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("retailer.gold.dim_date")

In [0]:
df_exchange = spark.table("retailer.silver.exchange_rates")

dim_exchange = df_exchange.select(
    "date",
    "currency",
    "exchange_rate"
)

dim_exchange.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("retailer.gold.dim_exchange_rates")

In [0]:
from pyspark.sql.functions import *

df_sales = spark.table("retailer.silver.sales")
df_products = spark.table("retailer.silver.products")
df_exchange = spark.table("retailer.silver.exchange_rates")

fact_sales = df_sales \
    .join(df_products, "product_key", "left") \
    .join(
        df_exchange,
        (df_sales.order_date == df_exchange.date) &
        (df_sales.currency_code == df_exchange.currency),
        "left"
    )

# Revenue Calculation
fact_sales = fact_sales.withColumn(
    "revenue_usd",
    col("quantity") * col("unit_price_usd") * col("exchange")
)

# Select columns (including delivery_date)
fact_sales = fact_sales.select(
    "order_number",
    "line_item",
    "order_date",
    "delivery_date",
    "customer_key",
    "product_key",
    "store_key",
    "quantity",
    "currency_code",
    "revenue_usd"
)

# ✅ Correct schema overwrite
fact_sales.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retailer.gold.fact_sales")

In [0]:
from pyspark.sql.functions import *

fact = spark.table("retailer.gold.fact_sales")
cust = spark.table("retailer.gold.dim_customers")
prod = spark.table("retailer.gold.dim_products")
store = spark.table("retailer.gold.dim_stores")
date = spark.table("retailer.gold.dim_date")

df_cube_base = fact \
    .join(cust, "customer_key", "left") \
    .join(prod, "product_key", "left") \
    .join(store, "store_key", "left") \
    .join(date, fact.order_date == date.date, "left")

In [0]:
df_cube = df_cube_base.cube(
    "year",
    "month",
    "continent",
    "category"
).agg(
    sum("revenue_usd").alias("total_revenue"),
    sum("quantity").alias("total_quantity")
)